# Vulnerability Detection - Entrainement CodeBERT

Fine-tuning de CodeBERT pour classifier du code comme vulnerable ou non.

## 1. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_from_disk
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

## 2. Verification GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Chargement du dataset

In [ ]:
dataset = load_from_disk("Defectors_Tokenized_Final")

for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    counts = Counter(labels)
    total = len(labels)
    print(f"{split:>12} : {total} exemples", end="  ->  ")
    for lbl, cnt in sorted(counts.items()):
        print(f"classe {lbl}: {cnt} ({cnt/total*100:.1f}%)", end="  ")
    print()

## 4. Preparation des donnees

In [ ]:
cols_remove = [c for c in dataset["train"].column_names if c not in ["input_ids", "attention_mask", "label"]]

train_dataset = dataset["train"].remove_columns(cols_remove)
eval_dataset = dataset["validation"].remove_columns(cols_remove)
test_dataset = dataset["test"].remove_columns(cols_remove)

train_dataset.set_format("torch")
eval_dataset.set_format("torch")
test_dataset.set_format("torch")

print(f"Train : {len(train_dataset)}, Val : {len(eval_dataset)}, Test : {len(test_dataset)}")
print(f"Colonnes : {train_dataset.column_names}")

## 5. Chargement du modele CodeBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2,
)
model.to(device)
print(f"Modele charge sur {device}")
print(f"Parametres : {sum(p.numel() for p in model.parameters()):,}")

## 6. Metriques

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

## 7. Configuration de l'entrainement

On utilise le Trainer standard sans poids de classes.
Le modele apprend directement la distribution naturelle du dataset.

In [ ]:
training_args = TrainingArguments(
    output_dir="./vulnerability_model_v3",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.06,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs_v3",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none",
)

steps = len(train_dataset) // training_args.per_device_train_batch_size
print(f"Steps/epoch : {steps}")
print(f"Total steps : {steps * int(training_args.num_train_epochs)}")
print(f"Temps estime : ~{steps * int(training_args.num_train_epochs) / 1.5 / 3600:.1f} heures")

## 8. Entrainement

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Entrainement en cours...")
train_result = trainer.train()
print(f"\nTermine en {train_result.metrics['train_runtime']:.0f}s")
print(f"Train loss : {train_result.metrics['train_loss']:.4f}")

## 9. Courbes de loss

In [ ]:
log_history = trainer.state.log_history
train_steps, train_losses = [], []
eval_steps, eval_losses, eval_accs, eval_f1s = [], [], [], []

for e in log_history:
    if "loss" in e and "eval_loss" not in e:
        train_steps.append(e["step"])
        train_losses.append(e["loss"])
    if "eval_loss" in e:
        eval_steps.append(e["step"])
        eval_losses.append(e["eval_loss"])
    if "eval_accuracy" in e:
        eval_accs.append(e["eval_accuracy"])
    if "eval_f1" in e:
        eval_f1s.append(e["eval_f1"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Suivi de l'entrainement", fontsize=14)

axes[0,0].plot(train_steps, train_losses, color="steelblue", alpha=0.7, linewidth=0.8)
axes[0,0].set_title("Training Loss")
axes[0,0].set_xlabel("Steps")
axes[0,0].set_ylabel("Loss")
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(eval_steps, eval_losses, color="coral", marker="o", markersize=4)
axes[0,1].set_title("Validation Loss")
axes[0,1].set_xlabel("Steps")
axes[0,1].set_ylabel("Loss")
axes[0,1].grid(True, alpha=0.3)

if eval_accs:
    axes[1,0].plot(eval_steps[:len(eval_accs)], eval_accs, color="seagreen", marker="o", markersize=4)
    axes[1,0].axhline(y=0.7, color="red", linestyle="--", alpha=0.5, label="Objectif 70%")
    axes[1,0].set_title("Validation Accuracy")
    axes[1,0].set_xlabel("Steps")
    axes[1,0].set_ylabel("Accuracy")
    axes[1,0].legend()
    axes[1,0].grid(True, alpha=0.3)

if eval_f1s:
    axes[1,1].plot(eval_steps[:len(eval_f1s)], eval_f1s, color="darkorchid", marker="o", markersize=4)
    axes[1,1].set_title("Validation F1-Score")
    axes[1,1].set_xlabel("Steps")
    axes[1,1].set_ylabel("F1")
    axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Training Loss vs Validation Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_steps, train_losses, color="steelblue", alpha=0.5, linewidth=0.8, label="Training Loss")
ax.plot(eval_steps, eval_losses, color="coral", marker="o", markersize=5, label="Validation Loss")
ax.set_title("Training Loss vs Validation Loss")
ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Evaluation sur le test set

In [ ]:
results = trainer.evaluate(test_dataset)
print("Resultats test set :")
print("=" * 40)
for k, v in sorted(results.items()):
    if k.startswith("eval_"):
        name = k.replace("eval_", "")
        print(f"  {name:<15} : {v:.4f}" if isinstance(v, float) else f"  {name:<15} : {v}")

## 12. Rapport de classification et matrice de confusion

In [ ]:
preds_output = trainer.predict(test_dataset)
pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("Rapport de classification :")
print("=" * 60)
print(classification_report(
    true_labels, pred_labels,
    target_names=["Non-vulnerable", "Vulnerable"],
    digits=4
))

cm = confusion_matrix(true_labels, pred_labels)
print("Matrice de confusion :")
print(f"                       Pred:Non-vuln  Pred:Vuln")
print(f"  Vrai: Non-vuln       {cm[0][0]:>7}       {cm[0][1]:>7}")
print(f"  Vrai: Vuln           {cm[1][0]:>7}       {cm[1][1]:>7}")

## 13. Visualisation matrice de confusion

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Matrice de confusion")
plt.colorbar(im, ax=ax)
classes = ["Non-vulnerable", "Vulnerable"]
ax.set_xticks([0, 1])
ax.set_xticklabels(classes)
ax.set_yticks([0, 1])
ax.set_yticklabels(classes)
for i in range(2):
    for j in range(2):
        color = "white" if cm[i,j] > cm.max()/2 else "black"
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", color=color, fontsize=14)
ax.set_ylabel("Vraie classe")
ax.set_xlabel("Classe predite")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Sauvegarde du modele

In [ ]:
trainer.save_model("./vulnerability_model_final")
print("Modele sauvegarde dans ./vulnerability_model_final")
print(f"\nResume :")
print(f"  Accuracy  : {results.get('eval_accuracy', 0):.4f}")
print(f"  F1-Score  : {results.get('eval_f1', 0):.4f}")
print(f"  Precision : {results.get('eval_precision', 0):.4f}")
print(f"  Recall    : {results.get('eval_recall', 0):.4f}")
print(f"  Loss      : {results.get('eval_loss', 0):.4f}")